# An IFC wall

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps

In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip
    await micropip.install(
        "https://ifcopenshell.github.io/wasm-wheels/"
        "ifcopenshell-0.8.4-cp313-cp313-pyodide_2025_0_wasm32.whl"
    )
else:
    %pip install -q ifcopenshell==0.8.4

import ifcopenshell
from ifcopenshell.api import run
from compas.geometry import Box, Frame
from compas_notebook import Viewer
from compas_notebook.config import Config

## Project, site, building, storey

In [ ]:
ifc = run("project.create_file", version="IFC4")
project = run("root.create_entity", ifc, ifc_class="IfcProject", name="First IFC")
site = run("root.create_entity", ifc, ifc_class="IfcSite", name="Site")
building = run("root.create_entity", ifc, ifc_class="IfcBuilding", name="Building")
storey = run("root.create_entity", ifc, ifc_class="IfcBuildingStorey", name="Ground floor")
for parent, child in [(project, site), (site, building), (building, storey)]:
    run("aggregate.assign_object", ifc, products=[child], relating_object=parent)

## Units and context

In [ ]:
units = [run("unit.add_si_unit", ifc, unit_type=kind) for kind in ["LENGTHUNIT", "AREAUNIT", "VOLUMEUNIT"]]
run("unit.assign_unit", ifc, units=units)
context = run("context.add_context", ifc, context_type="Model")
body = run("context.add_context", ifc, context_type="Model",
           context_identifier="Body", target_view="MODEL_VIEW", parent=context)

## Wall

In [ ]:
length, thickness, height = 4.0, 0.2, 3.0
wall = run("root.create_entity", ifc, ifc_class="IfcWall", name="Wall A")
shape = run("geometry.add_wall_representation", ifc, context=body,
            length=length, height=height, thickness=thickness)
run("geometry.assign_representation", ifc, product=wall, representation=shape)
run("geometry.edit_object_placement", ifc, product=wall)
run("spatial.assign_container", ifc, products=[wall], relating_structure=storey)
wall

## Preview

In [ ]:
preview = Box(length, thickness, height, frame=Frame([length / 2, thickness / 2, height / 2]))
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [7, -8, 5]
config.view.camera.target = [length / 2, 0, height / 2]
viewer = Viewer(config=config)
viewer.scene.add(preview.to_mesh(), color="#5552bb", show_edges=True)
viewer.show()

## Save and load

In [ ]:
ifc.write("first-wall.ifc")
loaded = ifcopenshell.open("first-wall.ifc")
loaded.by_type("IfcWall")